# Field Scan : adoption d'EfficientNet-B2 (seuil de rejet, réglages, contrôle sur les tests)

Le réseau a été choisi par la règle fixée à l'avance (expérience 1, `decision_reseau.json`). Ce carnet :
1. règle le **seuil de rejet** de B2 sur la validation (probabilité maximale, 95 % des feuilles connues acceptées, 4 vues), comme pour B0 ;
2. écrit le livrable : `maladies_b2.onnx` et son `entrainement_maladies.json` (classes, vues, normalisation, seuil) ;
3. fait passer B2 **une fois** sur les tests, avec la décision en deux étapes de `run.py`, à titre de **contrôle a posteriori** : le choix est déjà fait, et ces tests ont déjà été consultés (résultats indicatifs).

**Avant de lancer** : Input = `field-scan-feuilles` **et** la sortie du carnet de l'expérience 1 (dossier `comparaison_reseaux`) ; GPU ; Internet ; Save & Run All. Durée : environ 10 minutes.

## 1. Données et reprise

In [ ]:
import os, glob, zipfile, shutil
SORTIE = '/kaggle/working/comparaison_reseaux'        # copie de travail des modèles repris
MODELE_B2 = '/kaggle/working/modele_maladies_b2'     # livrable : ONNX + entrainement_maladies.json
REPRENDRE = True                                         # relire les modèles déjà entraînés au lieu de les réentraîner
os.makedirs(SORTIE, exist_ok=True)

# Données : index.csv dans un jeu de données attaché ; sinon, le zip est décompressé dans /tmp (hors des résultats)
index_trouves = [p for p in glob.glob('/kaggle/input/**/index.csv', recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), 'images'))]
if index_trouves:
    DONNEES = os.path.dirname(index_trouves[0])
else:
    zips = glob.glob('/kaggle/input/**/feuilles_pret*.zip', recursive=True)
    assert zips, "Aucun index.csv ni feuilles_pret*.zip dans /kaggle/input : ajoute le jeu de données (panneau Input)"
    DONNEES = '/tmp/feuilles'
    if not os.path.exists(f'{DONNEES}/index.csv'):
        with zipfile.ZipFile(zips[0]) as z:          # « \\ » remplacé par « / » si le zip vient de PowerShell
            for m in z.infolist():
                chemin = m.filename.replace('\\', '/')
                if chemin.endswith('/'):
                    continue
                cible = os.path.join(DONNEES, chemin)
                os.makedirs(os.path.dirname(cible), exist_ok=True)
                with z.open(m) as src, open(cible, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
print('Données :', DONNEES)

# Reprise : résultats d'une exécution précédente attachés en entrée
# (B0 : sortie du carnet d'entraînement ; autres réseaux : sortie d'une exécution précédente de ce carnet)
for anciens in glob.glob('/kaggle/input/**/modeles_maladies_perou', recursive=True) + glob.glob('/kaggle/input/**/comparaison_reseaux', recursive=True):
    for f in os.listdir(anciens):
        if not os.path.exists(os.path.join(SORTIE, f)):
            shutil.copy(os.path.join(anciens, f), SORTIE)
    print('Reprise depuis', anciens, ':', sorted(os.listdir(SORTIE)))

import pandas as pd
index = pd.read_csv(f'{DONNEES}/index.csv')
index = index[index.retenue].reset_index(drop=True)
assert all(os.path.exists(f'{DONNEES}/{c}') for c in index.chemin.sample(200, random_state=0)), "images manquantes"
print(index.groupby(['ensemble', 'classe']).size().unstack(fill_value=0))

## 2. Versions

In [ ]:
import torch, torchvision, platform, random, time, json, numpy as np
GRAINE = 2026
def graine():
    random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
graine()
VERSIONS = dict(torch=torch.__version__, torchvision=torchvision.__version__, python=platform.python_version(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')
print(VERSIONS)
assert torch.cuda.is_available(), "Pas de GPU : Settings > Accelerator > GPU T4 x2 ou GPU P100"
DEVICE = 'cuda'

## 3. Jeux de données

In [ ]:
import io
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

CLASSES = ['cercosporiose_noire', 'fusariose', 'saine']
MOYENNE, ECART = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
NORMALISATION = T.Normalize(MOYENNE, ECART)
moy, ec = torch.tensor(MOYENNE)[:, None, None], torch.tensor(ECART)[:, None, None]   # pour réafficher les images

class JPEGAleatoire:
    '''Recompression JPEG de qualité aléatoire, pour imiter des appareils différents.'''
    def __init__(self, p=0.5, qualites=(30, 95)): self.p, self.q = p, qualites
    def __call__(self, im):
        if random.random() > self.p: return im
        b = io.BytesIO(); im.save(b, 'JPEG', quality=random.randint(*self.q)); b.seek(0)
        return Image.open(b).convert('RGB')

class RecadrageCentre:
    def __init__(self, part): self.part = part
    def __call__(self, im):
        l, h = im.size; cl, ch = int(l * self.part), int(h * self.part)
        return im.crop(((l - cl) // 2, (h - ch) // 2, (l + cl) // 2, (h + ch) // 2))

def augmentations(recadrage_min, masquage):
    etapes = [JPEGAleatoire(), T.RandomResizedCrop(224, scale=(recadrage_min, 1.0)), T.RandomHorizontalFlip(),
              T.RandomVerticalFlip(), T.ColorJitter(0.3, 0.3, 0.3, 0.03),
              T.RandomApply([T.GaussianBlur(5, (0.1, 1.5))], p=0.2), T.ToTensor(), NORMALISATION]
    if masquage: etapes.append(T.RandomErasing(p=0.25, scale=(0.02, 0.15)))
    return T.Compose(etapes)

T_V1, T_V2 = augmentations(0.3, False), augmentations(0.15, True)
T_ENTIERE = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(), NORMALISATION])
T_CENTRE = T.Compose([RecadrageCentre(0.6), T.Resize(224), T.CenterCrop(224), T.ToTensor(), NORMALISATION])

class Feuilles(Dataset):
    '''Images de l'index ; étiquette = rang dans `classes`, ou -1 pour une classe inconnue.'''
    def __init__(self, lignes, transformation, classes=CLASSES):
        self.lignes, self.t, self.classes = lignes.reset_index(drop=True), transformation, classes
    def __len__(self): return len(self.lignes)
    def __getitem__(self, i):
        r = self.lignes.iloc[i]
        y = self.classes.index(r.classe) if r.classe in self.classes else -1
        return self.t(Image.open(f'{DONNEES}/{r.chemin}').convert('RGB')), y

def lignes(ensemble, classes=None):
    l = index[index.ensemble == ensemble]
    return l if classes is None else l[l.classe.isin(classes)]

def chargeur(lignes_, transformation, classes=CLASSES, melanger=False, taille=64):
    return DataLoader(Feuilles(lignes_, transformation, classes), batch_size=taille, shuffle=melanger,
                      num_workers=2, pin_memory=True, drop_last=melanger)

L_VAL = chargeur(lignes('validation'), T_ENTIERE)
L_VAL_CENTRE = chargeur(lignes('validation'), T_CENTRE)
print(len(lignes('entrainement')), 'images d’entraînement |', len(L_VAL.dataset), 'de validation')

## 4. Réseaux

In [ ]:
import torch.nn as nn
import torchvision.models as tvm
from sklearn.metrics import f1_score, confusion_matrix

ARCHITECTURES = {
    'efficientnet_b0': dict(fabrique=tvm.efficientnet_b0, poids=tvm.EfficientNet_B0_Weights.DEFAULT, lr=3e-4),
    'efficientnet_b2': dict(fabrique=tvm.efficientnet_b2, poids=tvm.EfficientNet_B2_Weights.DEFAULT, lr=3e-4),
    'convnext_tiny':   dict(fabrique=tvm.convnext_tiny,   poids=tvm.ConvNeXt_Tiny_Weights.DEFAULT,   lr=1e-4),
}

def nom_fichier(arch, simu=None):
    '''B0 reprend les fichiers du carnet d'entraînement (meilleur_v2.pt, meilleur_simu_sans_*.pt).'''
    if arch == 'efficientnet_b0':
        return 'v2' if simu is None else f'simu_sans_{simu}'
    return arch if simu is None else f'{arch}_simu_sans_{simu}'

def creer_modele(arch, n_classes=len(CLASSES), pre_entraine=True):
    a = ARCHITECTURES[arch]
    m = a['fabrique'](weights=a['poids'] if pre_entraine else None)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_classes)
    return m

def caracteristiques_et_logits(m, x):
    h = m.avgpool(m.features(x))
    if isinstance(m, tvm.ConvNeXt):                 # tête : LayerNorm2d, aplatissement, couche linéaire
        f = m.classifier[1](m.classifier[0](h))
        return f, m.classifier[2](f)
    f = torch.flatten(h, 1)
    return f, m.classifier(f)

@torch.no_grad()
def predire(m, chargeur_):
    m.eval(); F, L, Y = [], [], []
    for x, y in chargeur_:
        with torch.autocast('cuda', dtype=torch.float16):
            f, l = caracteristiques_et_logits(m, x.to(DEVICE, non_blocking=True))
        F.append(f.float().cpu()); L.append(l.float().cpu()); Y.append(y)
    return torch.cat(F), torch.cat(L), torch.cat(Y)

VUES = {'identite': lambda x: x, 'miroir_horizontal': lambda x: torch.flip(x, [3]),
        'miroir_vertical': lambda x: torch.flip(x, [2]), 'rotation_180': lambda x: torch.flip(x, [2, 3])}

@torch.no_grad()
def logits_4vues(m, chargeur_):
    m.eval(); L, Y = [], []
    for x, y in chargeur_:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            L.append(torch.stack([caracteristiques_et_logits(m, f(x))[1].float() for f in VUES.values()]).mean(0).cpu())
        Y.append(y)
    return torch.cat(L), torch.cat(Y)

def f1(L, Y):
    return round(float(f1_score(Y.numpy(), L.argmax(1).numpy(), average='macro')), 4)

def bilan(L, Y):
    m_ = confusion_matrix(Y.numpy(), L.argmax(1).numpy(), labels=range(len(CLASSES)))
    d = {f'rappel_{c}': round(float(m_[k, k] / m_[k].sum()), 3) for k, c in enumerate(CLASSES)}
    s = CLASSES.index('saine')
    d['fausses_alertes_saines'] = round(float((m_[s].sum() - m_[s, s]) / m_[s].sum()), 3)
    d['f1_macro'] = f1(L, Y)
    return d

PERTE = nn.CrossEntropyLoss(label_smoothing=0.1)
RECETTE = dict(epoques=12, lot=64, decroissance_poids=1e-4, lissage_etiquettes=0.1, programme='cosinus')

def entrainer(nom, arch, transformation, critere, classes=CLASSES, epoques=RECETTE['epoques']):
    '''critere : 'moyenne' (moyenne des F1 entières / recadrées) ou None (dernière époque : inconnues simulées).'''
    chemin, chemin_h = f'{SORTIE}/meilleur_{nom}.pt', f'{SORTIE}/historique_{nom}.json'
    m = creer_modele(arch, len(classes)).to(DEVICE)
    if REPRENDRE and os.path.exists(chemin) and os.path.exists(chemin_h):
        m.load_state_dict(torch.load(chemin)); h = json.load(open(chemin_h))
        print(f'{nom} : repris d’une exécution précédente (époque retenue {h["epoque_retenue"]})'); return m, h
    graine()
    L_tr = chargeur(lignes('entrainement', classes), transformation, classes, melanger=True)
    opt = torch.optim.AdamW(m.parameters(), lr=ARCHITECTURES[arch]['lr'], weight_decay=RECETTE['decroissance_poids'])
    prog = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epoques * len(L_tr))
    ech = torch.amp.GradScaler('cuda'); historique, meilleur = [], (-1, None)
    for ep in range(1, epoques + 1):
        m.train(); debut, total, n = time.time(), 0.0, 0
        for x, y in L_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                perte = PERTE(m(x), y)
            ech.scale(perte).backward(); ech.step(opt); ech.update(); prog.step()
            total += perte.item() * len(y); n += len(y)
        ligne = dict(epoque=ep, perte=round(total / n, 4))
        if critere is not None:
            _, Le, Ye = predire(m, L_VAL); _, Lr, Yr = predire(m, L_VAL_CENTRE)
            ligne.update(f1_entieres=f1(Le, Ye), f1_recadrages=f1(Lr, Yr))
            ligne['critere'] = round((ligne['f1_entieres'] + ligne['f1_recadrages']) / 2, 4)
        ligne['duree_s'] = round(time.time() - debut); historique.append(ligne); print(nom, ligne)
        valeur = ligne.get('critere', ep)
        if valeur > meilleur[0]:
            meilleur = (valeur, ep); torch.save(m.state_dict(), chemin)
    h = dict(historique=historique, epoque_retenue=meilleur[1], critere=critere, architecture=arch, lr=ARCHITECTURES[arch]['lr'])
    json.dump(h, open(chemin_h, 'w'), indent=2)
    m.load_state_dict(torch.load(chemin))
    print(f'{nom} : époque retenue {meilleur[1]}'); return m, h

## 5. Seuil de rejet de B2 (validation) et livrable

In [ ]:
import shutil
os.makedirs(MODELE_B2, exist_ok=True)
ARCH = 'efficientnet_b2'
m = creer_modele(ARCH).to(DEVICE)
m.load_state_dict(torch.load(f'{SORTIE}/meilleur_{ARCH}.pt')); m.eval()
L_val, Y_val = logits_4vues(m, L_VAL)
P_val = torch.softmax(L_val, 1)
conf_val = P_val.max(1).values
SEUIL = float(torch.quantile(conf_val, 0.05))                 # 95 % des feuilles connues acceptées
acceptees = conf_val >= SEUIL
exactitude_acceptees = float((P_val.argmax(1)[acceptees] == Y_val[acceptees]).float().mean())
print(f'Seuil de rejet B2 (probabilité maximale) : {SEUIL:.4f} | acceptées {float(acceptees.float().mean()):.3f} | '
      f'exactitude des acceptées {exactitude_acceptees:.4f}')
shutil.copy(f'{SORTIE}/maladies_{ARCH}.onnx', f'{MODELE_B2}/maladies_b2.onnx')
infos = dict(architecture=ARCH, classes=CLASSES, vues=list(VUES), entree=dict(taille=224, petit_cote=256),
             normalisation=dict(moyenne=[0.485, 0.456, 0.406], ecart_type=[0.229, 0.224, 0.225]),
             rejet=dict(methode='proba_max', seuils=dict(proba_max=round(SEUIL, 6)), acceptation_cible=0.95,
                        exactitude_acceptees_validation=round(exactitude_acceptees, 4)),
             validation=bilan(L_val, Y_val), choix='expérience 1 : règle fixée à l’avance (decision_reseau.json)',
             versions=VERSIONS)
json.dump(infos, open(f'{MODELE_B2}/entrainement_maladies.json', 'w'), indent=2, ensure_ascii=False)
print(sorted(os.listdir(MODELE_B2)))

## 6. Contrôle a posteriori sur les tests (indicatif)

In [ ]:
@torch.no_grad()
def reponses(ensemble, classe):
    L, _ = logits_4vues(m, chargeur(lignes(ensemble, [classe]), T_ENTIERE, CLASSES))
    P = torch.softmax(L, 1)
    ic, if_, is_ = CLASSES.index('cercosporiose_noire'), CLASSES.index('fusariose'), CLASSES.index('saine')
    p_malade = P[:, ic] + P[:, if_]
    sur = P.max(1).values >= SEUIL
    maladie = np.where((P[:, ic] >= P[:, if_]).numpy(), 'cercosporiose_noire', 'fusariose')
    rep = np.where((p_malade < 0.5).numpy(), np.where(sur.numpy(), 'saine', 'je_ne_sais_pas'),
                   np.where(sur.numpy(), maladie, 'malade_non_identifiee'))
    return pd.Series(rep).value_counts(normalize=True).round(3).to_dict(), len(rep)

lignes_tests = []
for ensemble, classes_ in (('test_perou', ['inconnue_cordana']),
                           ('test', ['cercosporiose_noire', 'saine', 'inconnue_cordana', 'inconnue_pestalotiopsis'])):
    for c in classes_:
        if len(lignes(ensemble, [c])) == 0:
            continue
        parts, n = reponses(ensemble, c)
        lignes_tests.append(dict(ensemble=ensemble, vraie_classe=c, n=n, **parts))
TESTS = pd.DataFrame(lignes_tests).fillna(0)
cols = ['ensemble', 'vraie_classe', 'n'] + [c for c in ['cercosporiose_noire', 'fusariose', 'malade_non_identifiee', 'je_ne_sais_pas', 'saine'] if c in TESTS]
TESTS = TESTS[cols]
print(TESTS.to_string(index=False))
TESTS.to_csv(f'{MODELE_B2}/tests_b2_indicatifs.csv', index=False)